# 05. "신호가 없다" 도 재현 가능해야 한다

> 2026-09-04 · 이동원 · 결론: [이슈 #105](https://github.com/devlee328288/Alpha_Stack/issues/105) ·
> 설계서 반영은 데이터파트 v3.4 (다음 PR)

공시 제목 155만 건에 감성 모델(`snunlp/KR-FinBert-SC`)을 돌려 확률 3칸을 만들고, 그 신호가
5일 방향과 관계가 있는지 카이제곱으로 쟀더니 **관계가 없었습니다** (Cramer's V = 0.026).

그 결과를 만든 세션은 이슈 #105 로 팀에 알렸지만 **재현 노트북과 시험을 남기지 않았고,
카이제곱을 계산한 코드가 저장소에 없습니다.** "신호가 있다" 만큼 "신호가 없다" 도 재현되어야
하는 결과입니다 — 그래야 나중에 누가 같은 칸을 피처로 넣으려 할 때 "왜 안 넣었나" 를 다시
잴 수 있습니다.

이 폴더의 규칙대로 **재현이 먼저**입니다. 원본 숫자를 그대로 낼 수 있어야 그다음 이야기가
성립합니다.

In [1]:
import bisect
import hashlib
import json
import os
import sqlite3
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(ROOT))
# HF 캐시 경고·진행 막대는 결과가 아니다 — 출력에서 뺀다.
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

from scripts.export_text_signal import HOLDOUT_START, KNOWN_RULE  # noqa: E402
from scripts.score_text_signal import MODEL_ID  # noqa: E402

pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 60)

# 읽기 전용으로 연다 — 이 노트북은 DB 를 바꾸지 않는다.
con = sqlite3.connect(f"file:{ROOT / 'data' / 'krx_cache.db'}?mode=ro", uri=True)
OUT = ROOT / "data" / "outbox" / "text_signal_20260904"
print("모델", MODEL_ID, "· known_at 규칙", KNOWN_RULE, "· 홀드아웃", HOLDOUT_START)

모델 snunlp/KR-FinBert-SC · known_at 규칙 rceptDt+1session · 홀드아웃 20240901


## 1. DB 는 제목 단위(18,600), 반출은 접수번호 단위(1,295,176) — 왜 둘인가

공시 제목은 정형 문구라 같은 문장이 평균 84번 반복됩니다. 행마다 추론하면 같은 문장을 84번씩
다시 읽는 셈이라 **고유 제목마다** 매기고(`text_signal` · 캐시), 팀원이 조인하는 축은 종목과
날짜라 **접수번호마다** 한 줄로 펴서 내보냅니다.

In [2]:
행수 = con.execute("SELECT COUNT(*) FROM dart_disclosure").fetchone()[0]
제목수 = con.execute(
    "SELECT COUNT(DISTINCT report_nm) FROM dart_disclosure "
    "WHERE report_nm IS NOT NULL AND report_nm <> ''").fetchone()[0]
print(f"공시 {행수:,}행 · 고유 제목 {제목수:,} ({제목수 / 행수:.1%}) "
      f"· 평균 반복 {행수 / 제목수:.0f}회")

display(pd.read_sql(
    "SELECT model_id, revision, COUNT(*) AS 행 FROM text_signal GROUP BY 1, 2", con))

# 가장 흔한 제목 10개가 공시 행의 몇 %를 덮나
흔한 = pd.read_sql(
    "SELECT report_nm AS 제목, COUNT(*) AS 건수 FROM dart_disclosure "
    "GROUP BY 1 ORDER BY 2 DESC LIMIT 10", con)
흔한["누적 비율"] = (흔한["건수"].cumsum() / 행수).map("{:.1%}".format)
흔한

공시 1,555,556행 · 고유 제목 18,600 (1.2%) · 평균 반복 84회


,model_id,revision,행
0,snunlp/KR-FinBert-SC,f8586286cc3161fb648e9fee09a456069fd846d0,18600


,제목,건수,누적 비율
0,임원ㆍ주요주주특정증권등소유상황보고서,197073,12.7%
1,주식등의대량보유상황보고서(일반),81066,17.9%
2,증권발행실적보고서,62221,21.9%
3,투자설명서(일괄신고),57171,25.6%
4,주식등의대량보유상황보고서(약식),46959,28.6%
5,최대주주등소유주식변동신고서,41398,31.2%
6,주주총회소집공고,38134,33.7%
7,주주총회소집결의,36738,36.0%
8,감사보고서제출,31898,38.1%
9,정기주주총회결과,31571,40.1%


### 반출본이 대장·HF 와 같은가

메모리의 데이터프레임이 아니라 **파일을 다시 열어** SHA-256 을 잽니다. HF 에 올라간 대장(2KB)도
받아 같은 해시인지 봅니다 — "올렸다" 와 "올라간 것이 이것이다" 는 다른 말입니다.

In [3]:
대장 = json.loads((OUT / "MANIFEST_text.json").read_text(encoding="utf-8"))
파일 = 대장["files"][0]
raw = (OUT / 파일["path"]).read_bytes()
df = pd.read_parquet(OUT / 파일["path"])
로컬해시 = hashlib.sha256(raw).hexdigest()

print(f"반출본 {len(df):,}행 × {df.shape[1]}칸 · 대장 {파일['rows']:,}행 "
      f"· {len(raw) / 2**20:.2f} MB")
print("대장 SHA  ", "✅ 같다" if 로컬해시 == 파일["sha256"] else "🔴 다르다")
print("접수 구간 ", df["rcept_dt"].min(), "~", df["rcept_dt"].max(),
      "· known_at 최대", df["known_at"].max(), f"(홀드아웃 {HOLDOUT_START} 앞)")
print("칸        ", list(df.columns))

try:
    from huggingface_hub import hf_hub_download
    p = hf_hub_download("qurious-quant/alphastack-krx-dev", "MANIFEST_text.json",
                        repo_type="dataset")
    hf대장 = json.loads(Path(p).read_text(encoding="utf-8"))
    hf해시 = hf대장["files"][0]["sha256"]
    print("HF 대장 SHA", "✅ 같다 — HF 에 올라간 것이 이 파일이다" if hf해시 == 로컬해시
          else f"🔴 다르다 (HF {hf해시[:16]}…)")
except Exception as e:  # noqa: BLE001 — 오프라인이면 이 대조만 건너뛴다
    print("HF 대장 대조 생략 —", type(e).__name__)

반출본 1,295,176행 × 13칸 · 대장 1,295,176행 · 11.88 MB
대장 SHA   ✅ 같다
접수 구간  20100104 ~ 20240829 · known_at 최대 20240830 (홀드아웃 20240901 앞)
칸         ['code', 'rcept_no', 'rcept_dt', 'known_at', 'known_rule', 'source', 'report_nm', 'model', 'model_rev', 'p_pos', 'p_neg', 'p_neu', 'text_sha256']


HF 대장 SHA ✅ 같다 — HF 에 올라간 것이 이 파일이다


## 2. 자르는 기준은 접수일이 아니라 `known_at` 이다 — 522행

`rcept_dt` 에는 시각이 없습니다. 15:00 접수 공시를 그날 신호로 쓰면 장 마감 30분 전에 알던 것이
되어 시세의 T+1 규약보다 앞섭니다. 그래서 **접수일 다음 거래일**부터 씁니다.

그러면 접수일이 개발구간 마지막 며칠인 행은 `known_at` 이 홀드아웃으로 넘어갑니다. 접수일로만
자르면 그 행이 "개발구간 자료" 얼굴로 들어오는데 실제로 쓸 수 있는 날은 봉인 구간입니다.

In [4]:
달력 = sorted(r[0] for r in con.execute(
    "SELECT bas_dd FROM trading_calendar WHERE market = 'ALL'"))


def 다음거래일(d):
    i = bisect.bisect_right(달력, d)
    return 달력[i] if i < len(달력) else None


접수 = pd.read_sql(f"""
    SELECT d.rcept_dt
      FROM dart_disclosure d
      JOIN text_signal t ON t.report_nm = d.report_nm AND t.model_id = ?
     WHERE d.rcept_dt < '{HOLDOUT_START}'
       AND d.stock_code IS NOT NULL AND d.stock_code <> ''""", con, params=(MODEL_ID,))
접수["known_at"] = 접수["rcept_dt"].map({d: 다음거래일(d) for d in 접수["rcept_dt"].unique()})

넘어감 = 접수[접수["known_at"] >= HOLDOUT_START]
달력밖 = int(접수["known_at"].isna().sum())
print(f"접수일로 자르면      {len(접수):,}행")
print(f"known_at 이 봉인 구간 {len(넘어감):,}행  ← 접수일 {sorted(넘어감['rcept_dt'].unique())}")
print(f"달력 밖(known_at 없음) {달력밖:,}행")
남는것 = len(접수) - len(넘어감) - 달력밖
print(f"남는 행 {남는것:,} = 반출본 {len(df):,} →",
      "✅ 같다" if 남는것 == len(df) else "🔴 다르다")

접수일로 자르면      1,295,698행
known_at 이 봉인 구간 522행  ← 접수일 ['20240830']
달력 밖(known_at 없음) 0행
남는 행 1,295,176 = 반출본 1,295,176 → ✅ 같다


## 3. 카이제곱 재현 — 원본 코드가 없어 설정을 되찾았다

이슈 #105 가 남긴 숫자는 이것입니다.

```
개발구간 1,265,118행 · chi2 = 1,698.3 · 자유도 4 · p < 1e-300 · Cramer's V = 0.026
기준선 상승 30.54% · 부정 28.83% · 중립 31.06% · 긍정 29.20%
```

자유도 4 = (3−1)×(3−1) 이니 방향도 세 갈래(하락·중립·상승)인데, **밴드가 얼마인지 · 수익률을
어느 가격에서 어느 가격으로 쟀는지가 적혀 있지 않습니다.** 수익률 정의 4가지 × 밴드 3가지를
전부 돌려 원본 숫자가 나오는 조합을 찾았습니다.

- 시세는 `adj_close`(수정주가)로 잽니다 — `close` 로 재면 액면분할이 −98% 로 읽힙니다 (#51)
- 5일 뒤 가격이 홀드아웃(20240901~)이면 뺍니다 — 미래 가격을 보지 않습니다
- 감성 라벨은 세 확률의 최댓값입니다 (동률이면 긍정 > 부정 > 중립)

In [5]:
시세 = pd.read_sql("SELECT code, bas_dd, adj_close, adj_open FROM daily_price", con)
종가 = 시세.set_index(["code", "bas_dd"])["adj_close"]
시가 = 시세.set_index(["code", "bas_dd"])["adj_open"]
print(f"시세 {len(시세):,}행 · 달력 {len(달력):,}일")

배열 = np.array(달력)
자리 = df["known_at"].map({d: i for i, d in enumerate(달력)}).astype(int).values
감성 = np.select(
    [(df["p_pos"] >= df["p_neg"]) & (df["p_pos"] >= df["p_neu"]),
     (df["p_neg"] > df["p_pos"]) & (df["p_neg"] >= df["p_neu"])],
    ["positive", "negative"], "neutral")


def 검정(기준가, 기준오프셋, 미래가, 미래오프셋, 밴드):
    """감성 × 5일 방향 카이제곱. 5일 뒤가 홀드아웃이거나 시세가 없으면 뺀다."""
    d0 = 배열[np.clip(자리 + 기준오프셋, 0, len(배열) - 1)]
    d1 = 배열[np.clip(자리 + 미래오프셋, 0, len(배열) - 1)]
    p0 = 기준가.reindex(pd.MultiIndex.from_arrays([df["code"].values, d0])).values
    p1 = 미래가.reindex(pd.MultiIndex.from_arrays([df["code"].values, d1])).values
    쓸것 = np.isfinite(p0) & np.isfinite(p1) & (p0 > 0) & (d1 < HOLDOUT_START)
    r = p1[쓸것] / p0[쓸것] - 1
    방향 = np.where(r > 밴드, "up", np.where(r < -밴드, "down", "flat"))
    표 = pd.crosstab(pd.Series(감성[쓸것], name="감성"), pd.Series(방향, name="5일 방향"))
    chi2, p, dof, _ = chi2_contingency(표)
    n = int(쓸것.sum())
    V = float(np.sqrt(chi2 / (n * (min(표.shape) - 1))))
    상승 = pd.Series(방향 == "up").groupby(감성[쓸것]).mean()
    요약 = {"n": n, "상승 비율": float((방향 == "up").mean()), "chi2": chi2, "dof": dof,
            "Cramer's V": V, "부정→상승": 상승.get("negative"),
            "중립→상승": 상승.get("neutral"), "긍정→상승": 상승.get("positive")}
    return 요약, 표, 방향, 쓸것


def 보기(요약):
    """숫자를 사람이 읽는 꼴로 — 표는 HTML 이 아니라 글자로 남긴다 (GitHub 에서도 보이게)."""
    return {"n": f"{요약['n']:,}", "상승 비율": f"{요약['상승 비율']:.2%}",
            "chi2": f"{요약['chi2']:,.1f}", "dof":요약["dof"],
            "Cramer's V": f"{요약["Cramer's V"]:.4f}",
            "부정→상승": f"{요약['부정→상승']:.2%}", "중립→상승": f"{요약['중립→상승']:.2%}",
            "긍정→상승": f"{요약['긍정→상승']:.2%}"}


# 원본과 맞은 설정 — 종가(known_at) → 종가(+5거래일) · 밴드 ±2%
# (features.stock_model_dataset 의 STOCK_NEUTRAL_BAND · 개별 종목 정본)
재현, 분할표, 방향, 쓸것 = 검정(종가, 0, 종가, 5, 0.02)
display(분할표)
원본 = {"n": 1_265_118, "상승 비율": 0.3054, "chi2": 1698.3, "dof": 4, "Cramer's V": 0.026,
        "부정→상승": 0.2883, "중립→상승": 0.3106, "긍정→상승": 0.2920}
pd.DataFrame({"이슈 #105 (원본)": 보기(원본), "이 노트북 (재현)": 보기(재현)}).T

시세 9,223,644행 · 달력 4,102일


5일 방향,down,flat,up
감성,,,
negative,35711,35661,28907
neutral,317914,323875,289206
positive,74224,91336,68278


,n,상승 비율,chi2,dof,Cramer's V,부정→상승,중립→상승,긍정→상승
이슈 #105 (원본),"1,265,118",30.54%,"1,698.3",4,0.0260,28.83%,31.06%,29.20%
이 노트북 (재현),"1,265,112",30.54%,"1,698.3",4,0.0259,28.83%,31.06%,29.20%


`chi2` 와 감성별 상승 비율이 소수점까지 같습니다. 행 수가 6 다른 것은 원본 코드를 못 본
채로 맞춘 한계입니다 — 설정을 못 맞췄다면 그렇게 적었을 텐데, 여기까지 맞으면 같은 계산입니다.

### 설정을 바꾸면 결론이 바뀌나

밴드를 0·1%·2% 로, 수익률을 종가→종가·시가→시가·전일종가→+4·시가→+4종가로 바꿔 12번 돌렸습니다.
**결론이 설정 하나에 매달려 있으면 그건 결론이 아닙니다.**

In [6]:
조합 = [
    ("종가(known_at) → 종가(+5)", 종가, 0, 종가, 5),
    ("시가(known_at) → 시가(+5)", 시가, 0, 시가, 5),
    ("종가(known_at−1) → 종가(+4)", 종가, -1, 종가, 4),
    ("시가(known_at) → 종가(+4)", 시가, 0, 종가, 4),
]
행들 = []
for 이름, a, ao, b, bo in 조합:
    for 밴드 in (0.0, 0.01, 0.02):
        요약, *_ = 검정(a, ao, b, bo, 밴드)
        행들.append({"수익률 정의": 이름, "밴드": f"±{밴드:.0%}", **요약})
탐색 = pd.DataFrame(행들)
V최소, V최대 = 탐색["Cramer's V"].min(), 탐색["Cramer's V"].max()
print(f"Cramer's V 범위 {V최소:.3f} ~ {V최대:.3f} — 어느 설정도 '작음'(0.1)의 절반에 못 미친다")
긍정이덜 = int((탐색["긍정→상승"] < 탐색["중립→상승"]).sum())
print(f"긍정이 중립보다 상승을 덜 맞히는 설정 {긍정이덜} / {len(탐색)}")
탐색.assign(**{
    "n": 탐색["n"].map("{:,}".format), "상승 비율": 탐색["상승 비율"].map("{:.1%}".format),
    "chi2": 탐색["chi2"].map("{:,.0f}".format),
    "Cramer's V": 탐색["Cramer's V"].map("{:.3f}".format),
    "부정→상승": 탐색["부정→상승"].map("{:.1%}".format),
    "중립→상승": 탐색["중립→상승"].map("{:.1%}".format),
    "긍정→상승": 탐색["긍정→상승"].map("{:.1%}".format),
}).drop(columns=["dof"])

Cramer's V 범위 0.012 ~ 0.039 — 어느 설정도 '작음'(0.1)의 절반에 못 미친다
긍정이 중립보다 상승을 덜 맞히는 설정 10 / 12


,수익률 정의,밴드,n,상승 비율,chi2,Cramer's V,부정→상승,중립→상승,긍정→상승
0,종가(known_at) → 종가(+5),±0%,"1,265,112",45.7%,"3,640",0.038,42.4%,45.8%,46.6%
1,종가(known_at) → 종가(+5),±1%,"1,265,112",37.8%,623,0.016,35.4%,38.2%,37.4%
2,종가(known_at) → 종가(+5),±2%,"1,265,112",30.5%,"1,698",0.026,28.8%,31.1%,29.2%
3,시가(known_at) → 시가(+5),±0%,"1,234,478",46.7%,363,0.012,44.6%,47.0%,46.5%
4,시가(known_at) → 시가(+5),±1%,"1,234,478",38.9%,"1,499",0.025,37.7%,39.4%,37.6%
5,시가(known_at) → 시가(+5),±2%,"1,234,478",31.7%,"2,893",0.034,31.3%,32.2%,29.5%
6,종가(known_at−1) → 종가(+4),±0%,"1,265,336",45.9%,"3,880",0.039,41.4%,46.1%,46.8%
7,종가(known_at−1) → 종가(+4),±1%,"1,265,336",38.2%,888,0.019,34.7%,38.6%,37.9%
8,종가(known_at−1) → 종가(+4),±2%,"1,265,336",31.0%,"1,936",0.028,28.8%,31.6%,29.8%
9,시가(known_at) → 종가(+4),±0%,"1,237,082",46.0%,341,0.012,43.8%,46.3%,45.9%


p 값은 12번 모두 0 에 가깝습니다. **표본이 126만이면 아주 작은 차이도 "통계적으로 유의" 합니다.**
그래서 효과크기를 봐야 하고, Cramer's V 는 어느 설정에서도 0.04 를 넘지 못합니다. 그리고 12번 중
대부분에서 **긍정 공시가 중립보다 상승을 덜 맞힙니다** — 방향조차 맞지 않습니다.

## 4. 제목별로 보면 큰 차이가 난다 — 그런데 감성이 아니라 시점이다

감성으로는 관계가 없는데, **제목별로** 5일 상승 비율을 재면 4.6% 에서 81.0% 까지 벌어집니다.
그 상위·하위가 전부 정기보고서입니다.

In [7]:
표 = pd.DataFrame({"제목": df["report_nm"].values[쓸것], "상승": 방향 == "up",
                   "접수월": pd.Series(df["rcept_dt"].values[쓸것]).str[:6].values,
                   "감성": 감성[쓸것]})
제목별 = (표.groupby("제목")
          .agg(건수=("상승", "size"), 상승비율=("상승", "mean"),
               감성=("감성", lambda s: s.mode().iat[0]))
          .query("건수 >= 1000").sort_values("상승비율"))
print(f"건수 1,000 이상인 제목 {len(제목별):,}종 · 상승 비율 "
      f"{제목별['상승비율'].min():.1%} ~ {제목별['상승비율'].max():.1%}")
양끝 = pd.concat([제목별.tail(5)[::-1], 제목별.head(5)])
양끝.assign(건수=양끝["건수"].map("{:,}".format), 상승비율=양끝["상승비율"].map("{:.1%}".format))

건수 1,000 이상인 제목 184종 · 상승 비율 4.6% ~ 81.0%


,건수,상승비율,감성
제목,,,
사업보고서 (2019.12),"1,929",81.0%,neutral
분기보고서 (2023.03),"2,225",63.0%,neutral
분기보고서 (2020.03),"1,953",51.7%,neutral
사업보고서 (2014.12),"1,487",51.6%,neutral
반기보고서 (2019.06),"1,865",49.8%,negative
반기보고서 (2015.06),"1,530",4.6%,negative
분기보고서 (2010.03),"1,214",7.5%,negative
분기보고서 (2019.09),"1,902",10.0%,negative
사업보고서 (2012.12),"1,354",11.1%,neutral


그런데 이 표만으로는 "감성 때문인지 시점 때문인지" 를 가를 수 없습니다 — "사업보고서 (2019.12)" 는
중립, "반기보고서 (2015.06)" 는 부정으로 매겨져 있으니 감성 차이로도 읽힙니다.

가르는 방법은 **같은 제목 가족 안에서** 보는 것입니다. "사업보고서 (YYYY.12)" 는 연도만 다를 뿐
문장이 같아 감성이 하나입니다. 그 안에서 상승 비율이 벌어진다면 그건 감성이 아닙니다.

In [8]:
def 가족(패턴):
    부분 = 표[표["제목"].str.match(패턴)]
    return (부분.groupby("제목")
            .agg(건수=("상승", "size"), 상승비율=("상승", "mean"),
                 감성=("감성", lambda s: " · ".join(sorted(s.unique()))),
                 접수월=("접수월", lambda s: s.mode().iat[0]))
            .sort_values("제목"))


for 이름, 패턴 in [("사업보고서 (YYYY.12)", r"^사업보고서 \(\d{4}\.12\)$"),
                 ("반기보고서 (YYYY.06)", r"^반기보고서 \(\d{4}\.06\)$")]:
    g = 가족(패턴)
    print(f"── {이름} — {len(g)}개 연도 · 감성 {set(g['감성'])} · "
          f"상승 비율 {g['상승비율'].min():.1%} ~ {g['상승비율'].max():.1%}")
    display(g.assign(건수=g["건수"].map("{:,}".format),
                     상승비율=g["상승비율"].map("{:.1%}".format)))

── 사업보고서 (YYYY.12) — 15개 연도 · 감성 {'neutral'} · 상승 비율 11.1% ~ 81.0%


,건수,상승비율,감성,접수월
제목,,,,
사업보고서 (2009.12),672,28.3%,neutral,201003
사업보고서 (2010.12),717,38.4%,neutral,201103
사업보고서 (2011.12),"1,243",13.4%,neutral,201203
사업보고서 (2012.12),"1,354",11.1%,neutral,201304
사업보고서 (2013.12),"1,415",33.4%,neutral,201403
사업보고서 (2014.12),"1,487",51.6%,neutral,201503
사업보고서 (2015.12),"1,584",35.7%,neutral,201603
사업보고서 (2016.12),"1,669",20.2%,neutral,201703
사업보고서 (2017.12),"1,751",23.5%,neutral,201803


── 반기보고서 (YYYY.06) — 15개 연도 · 감성 {'neutral', 'negative'} · 상승 비율 4.6% ~ 49.8%


,건수,상승비율,감성,접수월
제목,,,,
반기보고서 (2010.06),"1,223",30.2%,negative,201008
반기보고서 (2011.06),"1,303",12.7%,negative,201108
반기보고서 (2012.06),"1,343",34.3%,negative,201208
반기보고서 (2013.06),"1,370",24.1%,negative,201308
반기보고서 (2014.06),"1,436",32.4%,negative,201408
반기보고서 (2015.06),"1,530",4.6%,negative,201508
반기보고서 (2016.06),"1,622",20.9%,negative,201608
반기보고서 (2017.06),"1,706",35.5%,negative,201708
반기보고서 (2018.06),"1,779",46.4%,negative,201808


"사업보고서 (YYYY.12)" 15개 연도는 **전부 중립**인데 상승 비율은 11.1% 에서 81.0% 까지 벌어집니다.
"사업보고서 (2019.12)" 는 **2020년 3월**에 제출됩니다 — 코로나 폭락 뒤 반등 구간입니다.
"반기보고서 (2015.06)" 는 **2015년 8월** — 중국발 폭락 구간입니다. 차이를 만든 것은 제목에 박힌
**연월이 가리키는 시장 시점**이지 문장의 뜻이 아닙니다.

한 가지 더 보입니다. "반기보고서 (YYYY.06)" 은 **2010~2019 가 부정, 2020~2024 가 중립**입니다 —
문장은 같고 연도 숫자만 다른데 모델의 라벨이 갈립니다. 감성 모델조차 뜻이 아니라 **표기에
흔들린다**는 뜻이고, 그 라벨을 피처로 쓰면 이 흔들림이 그대로 들어옵니다.

제목을 원-핫이나 임베딩으로 넣으면 모델은 감성이 아니라 **"지금이 몇 년 몇 월인가"** 를 외웁니다.
개발구간 안에서는 성능이 오르는 것처럼 보이고 홀드아웃에서 무너집니다. 반출 파일에 `report_nm` 을
그대로 둔 이유는 사람이 확인하기 위해서지 피처로 쓰라는 뜻이 아닙니다 — 카드에 그렇게 적었습니다.

## 5. 설계서 §2.7 게이트(라벨 셋 다 15~45%)는 왜 안 썼나

In [9]:
from evaluation.horizon import CLASS_BALANCE_RANGE  # 읽기만 — 강민석 파트

분포 = pd.Series(감성).value_counts(normalize=True).reindex(["negative", "neutral", "positive"])
display(분포.map("{:.1%}".format).to_frame("비율"))
lo, hi = CLASS_BALANCE_RANGE
print("게이트", CLASS_BALANCE_RANGE, "→",
      "✅ 통과" if all(lo <= v <= hi for v in 분포)
      else "미달 — 공시는 원래 대부분이 중립인 자료라 이 전제가 안 맞는다. 카이제곱으로 갈음했다.")

,비율
negative,8.3%
neutral,73.4%
positive,18.3%


게이트 (0.15, 0.45) → 미달 — 공시는 원래 대부분이 중립인 자료라 이 전제가 안 맞는다. 카이제곱으로 갈음했다.


그 게이트는 **라벨(5일 방향)** 이 한쪽으로 쏠리지 않았는지 보는 것이지, 피처의 분포를 보는 것이
아닙니다. 공시는 원래 대부분이 중립(73%)이라 셋 다 15~45% 라는 전제 자체가 이 자료에 맞지
않습니다. 그래서 "분포가 고른가" 대신 **"바깥 기준(5일 방향)과 관계가 있는가"** 를 물었고, 그 답이
카이제곱입니다.

## 6. 오늘 배운 것

**p 값은 표본 크기를 따라간다. 효과크기를 봐라.** 126만 행이면 p < 1e-300 은 아무것도 말해 주지
않는다. Cramer's V 0.026 이 말한다 — 그리고 설정을 12번 바꿔도 0.04 를 못 넘는다.

**"신호가 없다" 도 재현 가능해야 한다.** 원본 코드가 저장소에 없어 설정을 되찾는 데 12번의 계산이
들었다. 정본이 없으면 재현이 없다 — 어제 배운 것과 같은 교훈이다.

**분포가 아니라 관계를 잰다.** 피처 분포 게이트는 이 자료에 안 맞았다. 맞지 않는 게이트를 억지로
통과시키는 대신, 무엇을 알고 싶었는지(바깥 기준과의 관계)로 돌아가 그에 맞는 검정을 고른다.

**제목은 시점을 외운다.** 같은 문장인데 연도에 따라 상승 비율이 갈렸다 — 감성이 아니라 연월이
가리키는 시장 국면이었다. 텍스트 피처는 "무엇이 적혀 있나" 만큼 "언제 적혔나" 가 새어 들어오는지
봐야 한다.

> [TIL 2026-09-04](../../docs/TIL/이동원/2026-09-04-신호가-없다도-재현-가능해야-한다.md)